# Финальный воспроизводимый пакет SBMTeam

**Команда Kaggle:** SBMTeam  
**Участники:** Peretiatko Vsevolod (`sevaperetiatko`) и Platon Usachev (`omnistudent`)  
**Основной submission:** `submission_NEW5_sqrt_s100_aux20.csv`  
**Дополнительный финальный вариант:** `submission_rescue_corrected_best.csv`

В этом ноутбуке оставлен только воспроизводимый финальный конвейер: подготовка признаков, обучение двух финальных ансамблей, несколько содержательных базовых моделей для сравнения, расчёт validation ROC-AUC и сохранение файлов предсказаний. Поисковые эксперименты и Optuna-подбор сюда не перенесены, чтобы запуск укладывался в ограничения финального пакета.


## Условия воспроизводимости

- Все пути относительны папки пакета; интернет и скрытые локальные файлы не используются.
- Фиксируется единый seed для Python, NumPy и всех стохастических моделей.
- Разбиение train/validation выполняется один раз со стратификацией.
- Обучаемые преобразования признаков строятся без утечки: признаки обучающей части формируются out-of-fold, а validation и test только преобразуются объектами, обученными на train.
- Дерево решений, случайный лес и градиентный бустинг получают расширенный набор engineered-признаков. Простой RBF SVC намеренно использует только исходные 512 признаков и служит понятной точкой отсчёта.
- Validation используется только для оценки готовых конфигураций и не участвует в обучении преобразований или моделей.
- После обучения заново создаются `final_submission.csv`, `submission_NEW5_sqrt_s100_aux20.csv` и `submission_rescue_corrected_best.csv`.


In [1]:
import os
from pathlib import Path

import pandas as pd
from IPython.display import display

from final_model_runner import MODEL_CONFIGURATIONS, SEED, run

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)
pd.set_option("display.width", 240)

print("Зафиксированный seed:", SEED)
print("Рабочая папка:", Path.cwd())
print("Доступных CPU-ядер:", os.cpu_count())


Зафиксированный seed: 1027309
Рабочая папка: c:\Users\peret\Desktop\Third year\ML\comp\comp 1\sbmt_kaggle\SBMTeam_final_package
Доступных CPU-ядер: 12


## Зафиксированные конфигурации моделей

Ниже перечислены десять сравниваемых решений. Первые четыре дают контраст между классическими деревьями и простым SVC, следующие четыре показывают вклад отдельных SVC-компонентов, а последние два являются финальными ансамблями. Параметры уже зафиксированы результатами предыдущих экспериментов и при финальном запуске не подбираются повторно.

Текст в ячейках переносится, а ограничение длины столбцов отключено, поэтому таблицу можно читать полностью и при необходимости прокручивать по горизонтали.


In [2]:
configuration_table = pd.DataFrame(MODEL_CONFIGURATIONS).rename(columns={
    "model": "Модель",
    "role": "Назначение",
    "parameters": "Параметры",
    "feature_set": "Признаки",
})

display(
    configuration_table.style
    .set_properties(**{
        "white-space": "pre-wrap",
        "text-align": "left",
        "vertical-align": "top",
    })
    .set_table_styles([
        {"selector": "th", "props": [("text-align", "left"), ("white-space", "nowrap")]},
        {"selector": "table", "props": [("width", "100%")]},
    ])
    .hide(axis="index")
)


Модель,Назначение,Параметры,Признаки
Дерево решений,"Простой интерпретируемый baseline: показывает, сколько структуры можно извлечь одним регуляризованным деревом без ансамблирования.","max_depth=5, min_samples_leaf=20","raw + leakage-safe whitened geometry, density, aggregates, coord, kNN и PCA128"
Случайный лес,Bagging-baseline для нелинейных разбиений; снижает дисперсию одиночного дерева и проверяет взаимодействия engineered-признаков.,"n_estimators=1000, max_depth=8, min_samples_leaf=40, min_samples_split=80, max_features=sqrt","raw + leakage-safe whitened geometry, density, aggregates, coord, kNN и PCA128"
Градиентный бустинг (GBDT),Последовательный ансамбль неглубоких деревьев; служит сильным контрастом kernel-методу на том же расширенном наборе признаков.,"n_estimators=2000, learning_rate=0.01, max_depth=3, min_samples_leaf=80, subsample=0.7","raw + leakage-safe whitened geometry, density, aggregates, coord, kNN и PCA128"
Простой raw RBF SVC,Контрольная SVC без engineered-блоков: отделяет эффект самого RBF-классификатора от эффекта разработанных представлений.,"log1p, StandardScaler, C=10, gamma=scale",только 512 исходных признаков
Fine6 SVC,"Исторический multi-kernel anchor, с которым сравнивались последующие исправления представлений и prediction blending.","C=6; additive weights raw=.68, geometry=.10, density=.10, aggregates=.05, knn=.05, coord=.02",исходный raw и original engineered-блоки
Corrected K0 SVC,Одиночная основная SVC после исправления whitening-представлений и gamma; показывает качество engineered-модели без блендинга.,C=3; Fine6 weights with corrected frozen block gammas,raw + whitened geometry/density/aggregates/coord + kNN
S100 SVC,"Главная одиночная SVC финального решения: использует sqrt-метрику raw и исправленные engineered-блоки, но ещё без auxiliary blend.","C=3; sqrt_raw=.68, geometry=.10, density=.10, aggregates=.05, knn=.05, coord=.02",sqrt(raw) + исправленные engineered-блоки
Вспомогательный clean_aux SVC,"Специально отличающаяся дополнительная модель: слабее основной отдельно, но даёт полезный complementary ranking для ансамбля.","C=30; pca128=.75, aggregates_white4=.25",PCA128 + whitened aggregates
Rescue T6,"Вторая финальная модель: консервативный rank-blend исторического Fine6, corrected K0 и complementary auxiliary SVC.","rank blend: Fine6=.40, corrected K0=.40, clean_aux=.20",три разных SVC-представления
NEW5 (финальная),Основная designated-модель: сочетает сильную sqrt-raw S100 и отличающийся PCA128 auxiliary ranking.,"rank blend: S100=.80, clean_aux=.20",sqrt/engineered SVC + PCA128 auxiliary SVC


### Как устроены два финальных ансамбля

Оба результата являются взвешенными смесями рангов предсказаний. Такой подход делает компоненты сопоставимыми по масштабу и устойчивее объединяет модели с разными распределениями score. Веса были выбраны на предыдущем исследовательском этапе и здесь остаются фиксированными.


In [3]:
FINAL_ENSEMBLES = {
    "NEW5 (финальная)": {
        "Ранг S100 SVC": 0.80,
        "Ранг clean_aux SVC": 0.20,
    },
    "Rescue T6": {
        "Ранг corrected K0 SVC": 0.55,
        "Ранг fine6 SVC": 0.25,
        "Ранг S100 SVC": 0.20,
    },
}

display(
    pd.DataFrame(FINAL_ENSEMBLES).fillna(0.0)
    .rename_axis("Компонент")
    .style.format("{:.2f}")
)


,NEW5 (финальная),Rescue T6
Компонент,,
Ранг S100 SVC,0.80,0.20
Ранг clean_aux SVC,0.20,0.00
Ранг corrected K0 SVC,0.00,0.55
Ранг fine6 SVC,0.00,0.25


## Обучение, оценка и генерация submission

Один вызов `run()` выполняет весь конвейер. Он загружает конкурсные CSV из относительных путей, создаёт leakage-safe признаки, обучает компоненты двух финальных ансамблей и десять моделей из таблицы сравнения, считает validation ROC-AUC, затем сохраняет вероятности для test.

Древесные модели используют исходные признаки вместе с геометрическими, плотностными, агрегатными, координатными, kNN- и PCA-блоками. Простой SVC использует только raw-признаки; это позволяет увидеть вклад разработанного feature engineering. Время каждой модели измеряется отдельно, а служебные затраты конвейера приводятся дополнительно.


In [4]:
result = run(output_dir=Path("."))

print("Созданы файлы:")
for output_path in result["output_paths"]:
    print(" -", output_path)


Созданы файлы:
 - submission_NEW5_sqrt_s100_aux20.csv
 - submission_rescue_corrected_best.csv
 - final_submission.csv


## Итоговое сравнение десяти моделей

В компактной итоговой таблице оставлены только четыре требуемых поля: название модели, validation ROC-AUC, время выполнения и seed. Подробные параметры, назначение и набор признаков уже приведены выше в таблице конфигураций, поэтому здесь они не дублируются.


In [5]:
experiment_table = result["experiment_table"].copy()

display(
    experiment_table.style
    .format({
        "Validation ROC-AUC": "{:.6f}",
        "Время, сек": "{:.2f}",
    })
    .hide(axis="index")
)

timing_names = {
    "preprocessing_seconds": "Подготовка данных и признаков",
    "final_components_seconds": "Компоненты финальных ансамблей",
    "comparison_models_seconds": "Сравнительные модели",
    "test_prediction_seconds": "Формирование test-предсказаний после fit",
    "total_seconds": "Полное время конвейера",
}
timing_table = pd.DataFrame([
    {"Этап": timing_names.get(name, name), "Время, сек": seconds}
    for name, seconds in result["timings"].items()
])
display(timing_table.style.format({"Время, сек": "{:.2f}"}).hide(axis="index"))


Модель,Validation ROC-AUC,"Время, сек",Seed
Дерево решений,0.904298,0.58,1027309
Случайный лес,0.923661,3.31,1027309
Градиентный бустинг (GBDT),0.948776,186.20,1027309
Простой raw RBF SVC,0.947423,3.12,1027309
Fine6 SVC,0.955038,41.96,1027309
Corrected K0 SVC,0.954082,41.96,1027309
S100 SVC,0.955038,18.03,1027309
Вспомогательный clean_aux SVC,0.938342,41.96,1027309
Rescue T6,0.955179,430.54,1027309
NEW5 (финальная),0.954936,430.54,1027309


Этап,"Время, сек"
Подготовка данных и признаков,176.98
rescue_components_fit_predict_seconds,41.96
s100_fit_predict_seconds,18.03
Полное время конвейера,430.54


## Версии среды

Таблица фиксирует версии Python и основных библиотек, от которых зависит воспроизводимость расчётов.


In [6]:
versions_table = pd.DataFrame(
    [{"Пакет": name, "Версия": version} for name, version in result["versions"].items()]
)
display(versions_table.style.hide(axis="index"))


Пакет,Версия
python,3.14.2
numpy,2.5.3
pandas,3.0.6
scipy,1.18.1
scikit-learn,1.9.1
joblib,1.6.0


## Автоматические проверки результата

В конце проверяется, что `row_id` совпадает с test в исходном порядке, вероятности конечны и находятся в диапазоне `[0, 1]`, а все три ожидаемых CSV действительно созданы. Эти проверки ловят наиболее опасные ошибки сборки submission до загрузки на Kaggle или Moodle.


In [7]:
assert result["row_id_exact"], "row_id в submission не совпадает с test.csv"
assert result["probabilities_valid"], "Предсказания содержат некорректные вероятности"
assert all(Path(path).is_file() for path in result["output_paths"]), "Созданы не все CSV-файлы"

print("Проверка row_id: пройдена")
print("Проверка вероятностей: пройдена")
print("Все ожидаемые CSV созданы: да")


Проверка row_id: пройдена
Проверка вероятностей: пройдена
Все ожидаемые CSV созданы: да
